# Creating a Simple Agent with Tracing

In [2]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
try:
    print(
        OpenAI()
        .responses.create(
            model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
        )
        .output_text
    )
except Exception as e:
    print(f"API Error: {type(e).__name__}")
    print(f"Details: {str(e)}")
    print("\nTo resolve: Check your OpenAI account billing at https://platform.openai.com/account/billing/overview")

We are up and running!


In [3]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [6]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy is arugula?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Very healthy. Arugula is low in calories but high in nutrients.
    
    Key points:
    - Vitamins/minerals: high in vitamin K, A, C; also folate, calcium, potassium.
    - Antioxidants: contains glucosinolates and carotenoids with potential anti-cancer benefits.
    - Fiber: contributes to daily fiber intake.
    - Low calories: great for salads and meals.
    
    Considerations:
    - If you’re on blood thinners (warfarin), keep vitamin K intake consistent.
    - Some people may be sensitive or allergic.
    
    Ways to use: add to salads, sandwiches, grain bowls, or lightly sauté as a side.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [7]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, convenient fruit. Per medium banana (about 118 g):
- Nutrients: ~400–450 mg potassium, ~0.3 g protein, vitamin B6, vitamin C, fiber (if with skin), and small amounts of several other nutrients.
- Benefits: supports blood pressure, helps digestion, provides quick energy from natural sugars, and adds fiber.
- Considerations: moderate sugar content; may raise blood sugar more than some berries. Kidney disease patients should monitor potassium intake.
- Bottom line: generally healthy as part of a balanced diet; pair with proteins/fats if you’re watching blood sugar.

_Good Job!_